# Autodesk Agentic AI — Opportunity-Level Inside Sales Decision Platform

**Beginner-friendly, end-to-end notebook**

### Objective

> Given the current state of an Autodesk sales opportunity, recommend the best next sales strategy and action to improve the probability of progressing or closing the opportunity.

The system uses:

- CRM / opportunity data
- Web engagement
- Last **maximum 10 actions**
- Days between actions
- Response to each action
- Rep who took each action
- Current rep workload
- Number of reps who handled the opportunity
- **Actions taken by each rep**
- Email / call notes when available
- Opportunity risk model
- Retention / uplift model
- NBA Offline RL / IQL model
- Autodesk Product Catalog RAG (OpenAI embeddings + local FAISS semantic retrieval)
- Customer 3-year license + usage adoption trend
- 4 agents
- LangGraph
- Human approval
- Memory / checkpointing
- Middleware
- Caching
- Guardrails
- Evaluation

## Two modes

The notebook is configured for the real LLM path:

```python
USE_LLM = True
```

That means the full workflow runs without an API key using deterministic fallback logic.

When you are ready for real LLM agents:

```python
USE_LLM = True
```

and provide `OPENAI_API_KEY`.

If a real LLM call fails, the notebook falls back to deterministic logic instead of crashing the workflow.### Production principle used in this version

For product matching we do **not** use rules such as:

```python
if "bim" in query:
    recommend("Revit")
```

Instead:

```text
Customer need
    ↓
Embedding
    ↓
FAISS semantic retrieval
    ↓
Grounded catalog evidence
    ↓
LLM Product Fit reasoning
```

Deterministic `if` checks are still valid for:
- missing data
- safety
- permissions
- guardrails
- routing
- execution validation

Those are business/control logic, not semantic product recommendation.

# Step 1 — Architecture

We use **4 true agents**.

## 1. Opportunity Intelligence Agent

Purpose:
Understand the opportunity and the customer's conversation.

Tools:
- `get_opportunity_tool`
- `get_conversation_tool`

LLM extracts:
- summary
- intent
- pain points
- objections
- requirements
- use case
- timeline
- mentioned products
- missing information

---

## 2. Product Fit & Solution Recommendation Agent

Purpose:
Check whether the customer's current / discussed Autodesk product fits the
actual business requirement.

Tool:
- `autodesk_catalog_rag_tool`

Retrieval backend in this notebook:
- OpenAI embeddings
- local FAISS vector index

There are **no hard-coded semantic mappings** such as
`BIM -> Revit`.

---

## 3. Decision / Sales Strategy Agent

Purpose:
Combine model signals and business context into the final sales strategy.

Tools:
- `opportunity_risk_tool`
- `retention_uplift_tool`
- `nba_iql_tool`
- `rep_workload_tool`
- `customer_adoption_trend_tool`

The adoption tool analyzes the customer's last 3 years of:
- license count
- usage tokens

and returns deterministic increasing / stable / declining signals.

---

## 4. Action Agent

Purpose:
Execute only the already-decided and approved action.

Tools:
- Email
- LinkedIn
- Call task
- Demo
- Nurture
- Wait
- CRM update

---

## End-to-end flow

```text
CRM + Email + Calls
        ↓
Opportunity Intelligence Agent
        ↓
OpportunityIntelligenceOutput
        ↓
Product Fit Agent
        │
        └── Autodesk Catalog RAG
             ↓
          Embedding
             ↓
           FAISS
        ↓
ProductRecommendationOutput
        ↓
Decision / Strategy Agent
        │
        ├── Risk
        ├── Uplift
        ├── NBA-IQL
        ├── Rep Workload
        └── 3-Year Adoption Trend
        ↓
Deterministic Guardrails
        ↓
Human Approval if needed
        ↓
Action Agent
        ↓
END
```

# Step 2 — Install Libraries

The LangChain/LangGraph versions are pinned so the notebook uses one consistent API.

For the local production-style RAG prototype we also install:

- `faiss-cpu` — local dense-vector similarity index
- `numpy` — vector handling

Important:
FAISS is the **local retrieval backend** for this notebook.
Later, we can replace only this backend with OpenSearch Serverless or another
managed vector database without redesigning the Product Fit Agent.

In [ ]:
%pip install -q \
    langchain==1.3.15 \
    langgraph==1.2.11 \
    langchain-openai==1.5.1 \
    faiss-cpu \
    numpy \
    pandas \
    pydantic

In [ ]:
import os
import re
import json
import logging
import getpass
from functools import lru_cache
from typing import Any, Dict, List, Literal, Optional
from typing_extensions import TypedDict

import pandas as pd
import numpy as np
from pydantic import BaseModel, Field

from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.documents import Document

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

# ----------------------------------------------------
# Main switches
# ----------------------------------------------------

# True = use real LLM agents.
# We keep a safe fallback only so the notebook is still debuggable.
USE_LLM = True

# False = auto-approve in the main demo.
# True  = pause the graph for human approval.
ENABLE_HITL = False

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.5")

# Securely request the key only when it is not already in the environment.
if USE_LLM and not os.getenv("OPENAI_API_KEY"):
    entered_key = getpass.getpass(
        "Enter OPENAI_API_KEY: "
    ).strip()

    if entered_key:
        os.environ["OPENAI_API_KEY"] = entered_key


USE_LLM_EFFECTIVE = (
    USE_LLM
    and bool(os.getenv("OPENAI_API_KEY"))
)

if USE_LLM and not USE_LLM_EFFECTIVE:
    print(
        "OPENAI_API_KEY not available -> "
        "live LLM agents cannot run."
    )

print("USE_LLM_EFFECTIVE =", USE_LLM_EFFECTIVE)
print("ENABLE_HITL       =", ENABLE_HITL)

# Step 3 — Create Sample CRM Data

For learning, pandas DataFrames simulate database/CRM tables.

Later these can be replaced by Salesforce, PostgreSQL, Snowflake, Databricks, etc.

In [ ]:
opportunity_df = pd.DataFrame([
    {
        "opportunity_id": "OPP_1001",
        "account_id": "ACC_001",
        "industry": "Architecture",
        "company_size": 500,
        "region": "US",
        "stage": "Evaluation",
        "deal_value": 250000.0,
        "expected_close_days": 45,
        "current_products": ["AutoCAD"],
        "product_interest": ["Revit"],
        "current_rep_id": "REP_02",
    },
    {
        "opportunity_id": "OPP_1002",
        "account_id": "ACC_002",
        "industry": "Manufacturing",
        "company_size": 200,
        "region": "India",
        "stage": "Discovery",
        "deal_value": 120000.0,
        "expected_close_days": 75,
        "current_products": ["AutoCAD"],
        "product_interest": ["Fusion"],
        "current_rep_id": "REP_03",
    },
])

opportunity_df

In [ ]:
engagement_df = pd.DataFrame([
    {
        "opportunity_id": "OPP_1001",
        "website_visits_30d": 12,
        "pricing_page_visits_30d": 3,
        "demo_requested": False,
    },
    {
        "opportunity_id": "OPP_1002",
        "website_visits_30d": 4,
        "pricing_page_visits_30d": 1,
        "demo_requested": False,
    },
])

engagement_df

In [ ]:
# Historical actions.
# We calculate gap_days from these dates later.

action_df = pd.DataFrame([
    {"opportunity_id": "OPP_1001", "action_date": "2026-07-20", "action": "email",        "response": "opened",    "rep_id": "REP_01"},
    {"opportunity_id": "OPP_1001", "action_date": "2026-07-23", "action": "call",         "response": "no_answer", "rep_id": "REP_01"},
    {"opportunity_id": "OPP_1001", "action_date": "2026-07-27", "action": "nurture",      "response": "clicked",   "rep_id": "REP_02"},
    {"opportunity_id": "OPP_1001", "action_date": "2026-07-29", "action": "email",        "response": "replied",   "rep_id": "REP_02"},
    {"opportunity_id": "OPP_1001", "action_date": "2026-08-02", "action": "linkedin_msg", "response": "viewed",    "rep_id": "REP_02"},

    {"opportunity_id": "OPP_1002", "action_date": "2026-07-10", "action": "email",        "response": "opened",    "rep_id": "REP_04"},
    {"opportunity_id": "OPP_1002", "action_date": "2026-07-18", "action": "nurture",      "response": "no_action", "rep_id": "REP_03"},
    {"opportunity_id": "OPP_1002", "action_date": "2026-08-01", "action": "call",         "response": "no_answer", "rep_id": "REP_03"},
])

action_df["action_date"] = pd.to_datetime(
    action_df["action_date"]
)

action_df = action_df.sort_values(
    ["opportunity_id", "action_date"]
).reset_index(drop=True)

action_df

In [ ]:
rep_df = pd.DataFrame([
    {
        "rep_id": "REP_01",
        "experience_months": 24,
        "conversion_rate": 0.25,
        "active_opportunities": 40,
        "product_specialization": "AEC",
    },
    {
        "rep_id": "REP_02",
        "experience_months": 36,
        "conversion_rate": 0.31,
        "active_opportunities": 35,
        "product_specialization": "AEC",
    },
    {
        "rep_id": "REP_03",
        "experience_months": 48,
        "conversion_rate": 0.35,
        "active_opportunities": 28,
        "product_specialization": "Manufacturing",
    },
    {
        "rep_id": "REP_04",
        "experience_months": 18,
        "conversion_rate": 0.22,
        "active_opportunities": 55,
        "product_specialization": "Manufacturing",
    },
])

rep_df

In [ ]:
# OPP_1001 has conversation data.
# OPP_1002 intentionally has no conversation row.

conversation_df = pd.DataFrame([
    {
        "opportunity_id": "OPP_1001",
        "email_text": (
            "We currently use AutoCAD for most design work. "
            "We are working on hospital projects and want better coordination "
            "between architecture and structural teams. We are evaluating BIM "
            "solutions, but pricing is important. We expect to decide next quarter."
        ),
        "call_notes": (
            "Customer mentioned approximately 60 designers across multiple locations. "
            "They want to understand how BIM can improve collaboration between teams."
        ),
    }
])

conversation_df

## Step 3.1 — Sample 3-Year License + Product Usage History

In production this data would usually come from:
- subscription / licensing tables
- product telemetry / usage tables
- feature store / warehouse

For the notebook we simulate **36 monthly observations per account**.

Important:

The LLM does **not** calculate whether usage is increasing or declining.

A deterministic analytics tool calculates:
- license trend
- usage-token trend
- recent year-over-year change
- usage per license
- overall adoption signal

This keeps numeric trend computation outside the LLM.

In [ ]:
# ------------------------------------------------------------
# 36 monthly observations: Sep-2023 through Aug-2026
# ------------------------------------------------------------

months = pd.date_range(
    start="2023-09-01",
    periods=36,
    freq="MS",
)

license_usage_rows = []

for i, month in enumerate(months):

    # ACC_001:
    # licenses increase while real usage declines.
    # This is useful for demonstrating an under-utilization signal.
    license_usage_rows.append({
        "account_id": "ACC_001",
        "month": month,
        "license_count": 120 + i,
        "usage_tokens": 120_000 - (1_200 * i),
    })

    # ACC_002:
    # both licenses and usage increase.
    license_usage_rows.append({
        "account_id": "ACC_002",
        "month": month,
        "license_count": 80 + int(i * 0.6),
        "usage_tokens": 65_000 + (1_100 * i),
    })


license_usage_df = pd.DataFrame(
    license_usage_rows
).sort_values(
    ["account_id", "month"]
).reset_index(drop=True)


license_usage_df.tail()

# Step 4 — Pydantic Schemas

Pydantic gives us validated, predictable contracts between tools, agents and graph nodes.

In [ ]:
ActionType = Literal[
    "email",
    "linkedin_msg",
    "nurture",
    "call",
    "demo_schedule",
    "wait",
]


class Engagement(BaseModel):
    website_visits_30d: int = Field(default=0, ge=0)
    pricing_page_visits_30d: int = Field(default=0, ge=0)
    demo_requested: bool = False


class ActionHistory(BaseModel):
    action_date: str
    action: ActionType
    gap_days: int = Field(ge=0)
    response: Optional[str] = None
    rep_id: Optional[str] = None


class RepContext(BaseModel):
    current_rep_id: str
    experience_months: int = Field(ge=0)
    conversion_rate: float = Field(ge=0.0, le=1.0)
    active_opportunities: int = Field(ge=0)
    product_specialization: Optional[str] = None

    num_reps_handled: int = Field(ge=0)
    handoff_count: int = Field(ge=0)
    current_rep_actions_on_opportunity: int = Field(ge=0)

    # User-requested feature:
    # count of actions each rep took on this opportunity.
    actions_per_rep: Dict[str, int] = Field(
        default_factory=dict
    )


class ConversationContext(BaseModel):
    email_text: Optional[str] = None
    call_notes: Optional[str] = None


class OpportunityState(BaseModel):
    opportunity_id: str
    account_id: str
    industry: Optional[str] = None
    company_size: Optional[int] = Field(default=None, ge=0)
    region: Optional[str] = None
    stage: Optional[str] = None
    deal_value: Optional[float] = Field(default=None, ge=0)
    expected_close_days: Optional[int] = Field(default=None, ge=0)

    current_products: List[str] = Field(default_factory=list)
    product_interest: List[str] = Field(default_factory=list)

    engagement: Engagement

    # Final sequential input keeps only latest max 10 actions.
    action_history: List[ActionHistory] = Field(
        default_factory=list,
        max_length=10,
    )

    rep_context: RepContext
    conversation_context: Optional[ConversationContext] = None


class OpportunityIntelligenceOutput(BaseModel):
    summary: str
    intent: Literal["low", "medium", "high", "unknown"]
    pain_points: List[str] = Field(default_factory=list)
    objections: List[str] = Field(default_factory=list)
    requirements: List[str] = Field(default_factory=list)
    timeline: Optional[str] = None
    use_case: Optional[str] = None

    # Product names explicitly mentioned in email/call/CRM conversation.
    # The live LLM must not invent product names here.
    mentioned_products: List[str] = Field(default_factory=list)

    missing_information: List[str] = Field(default_factory=list)


class CustomerAdoptionTrendOutput(BaseModel):
    """
    Deterministic 3-year license / usage trend output.
    """

    opportunity_id: str
    account_id: str
    months_analyzed: int = Field(ge=1)

    license_trend: Literal[
        "increasing",
        "stable",
        "declining",
    ]

    usage_trend: Literal[
        "increasing",
        "stable",
        "declining",
    ]

    license_annualized_slope_pct: float
    usage_annualized_slope_pct: float

    recent_license_yoy_pct: Optional[float] = None
    recent_usage_yoy_pct: Optional[float] = None

    latest_license_count: int
    latest_usage_tokens: float
    latest_usage_per_license: float

    overall_adoption_signal: Literal[
        "expanding",
        "stable",
        "contracting",
        "underutilization_risk",
        "concentrated_usage",
        "mixed",
    ]


class ProductRecommendationOutput(BaseModel):
    # Product(s) customer currently uses.
    current_products: List[str] = Field(default_factory=list)

    # Product(s) customer is currently asking about / considering.
    current_product_interest: List[str] = Field(default_factory=list)

    # What the customer is actually trying to achieve.
    customer_need: str

    # GOOD_FIT:
    #   Current/requested product matches the stated need.
    #
    # POSSIBLE_MISMATCH:
    #   Current/requested product may not be the best fit.
    #
    # NEEDS_DISCOVERY:
    #   Not enough information to recommend confidently.
    fit_status: Literal[
        "GOOD_FIT",
        "POSSIBLE_MISMATCH",
        "NEEDS_DISCOVERY",
    ]

    # Best-fit Autodesk product based on catalog evidence.
    recommended_product: str

    # Explanation of product fit.
    reason: str

    # Other relevant Autodesk products.
    alternative_products: List[str] = Field(default_factory=list)

    # Retrieved catalog evidence.
    supporting_evidence: List[str] = Field(default_factory=list)

    # Information still required from the customer.
    missing_information: List[str] = Field(default_factory=list)

    # What the sales rep should do with this product-fit insight.
    recommended_next_step: str


class SalesStrategyOutput(BaseModel):
    recommended_action: ActionType
    recommended_product: str
    strategy: str
    reason: str
    talking_points: List[str] = Field(default_factory=list)
    discovery_questions: List[str] = Field(default_factory=list)
    requires_human_approval: bool = False


class ActionAgentOutput(BaseModel):
    action: ActionType
    status: Literal[
        "success",
        "pending_approval",
        "failed",
        "no_action",
        "rejected",
    ]
    message: str


class GuardrailResult(BaseModel):
    passed: bool
    reasons: List[str] = Field(default_factory=list)
    force_human_approval: bool = False

# Step 5 — Build Tools

The LLM receives controlled tools instead of unrestricted database/API access.

In [ ]:
def _calculate_handoff_count(
    rep_sequence: List[str],
) -> int:
    """
    Count actual changes in rep ownership/action sequence.

    Example:
    [R1, R1, R2, R2, R1] -> 2 handoffs
    """

    if len(rep_sequence) < 2:
        return 0

    return sum(
        rep_sequence[i] != rep_sequence[i - 1]
        for i in range(1, len(rep_sequence))
    )


def get_opportunity_data(
    opportunity_id: str,
) -> dict:
    """
    Build one opportunity state from CRM tables.
    """

    # 1. Opportunity
    rows = opportunity_df[
        opportunity_df["opportunity_id"] == opportunity_id
    ]

    if rows.empty:
        return {
            "error": f"Opportunity {opportunity_id} not found"
        }

    opp = rows.iloc[0].to_dict()

    # 2. Engagement
    eng_rows = engagement_df[
        engagement_df["opportunity_id"] == opportunity_id
    ]

    if eng_rows.empty:
        engagement = {
            "website_visits_30d": 0,
            "pricing_page_visits_30d": 0,
            "demo_requested": False,
        }
    else:
        engagement = eng_rows.iloc[0].to_dict()
        engagement.pop("opportunity_id", None)

    # 3. ALL actions for rep aggregates
    all_actions = action_df[
        action_df["opportunity_id"] == opportunity_id
    ].sort_values("action_date").copy()

    if all_actions.empty:
        actions_per_rep = {}
        rep_sequence = []
    else:
        actions_per_rep = (
            all_actions["rep_id"]
            .value_counts()
            .to_dict()
        )
        rep_sequence = (
            all_actions["rep_id"]
            .astype(str)
            .tolist()
        )

    num_reps_handled = len(set(rep_sequence))
    handoff_count = _calculate_handoff_count(
        rep_sequence
    )

    # 4. Latest max 10 actions
    recent = all_actions.tail(10).copy()

    if recent.empty:
        action_history = []
    else:
        recent["gap_days"] = (
            recent["action_date"]
            .diff()
            .dt.days
            .fillna(0)
            .astype(int)
            .clip(lower=0)
        )

        recent["action_date"] = (
            recent["action_date"]
            .dt.strftime("%Y-%m-%d")
        )

        action_history = recent[
            [
                "action_date",
                "action",
                "gap_days",
                "response",
                "rep_id",
            ]
        ].to_dict(orient="records")

    # 5. Current rep
    current_rep_id = opp["current_rep_id"]

    rep_rows = rep_df[
        rep_df["rep_id"] == current_rep_id
    ]

    rep = (
        rep_rows.iloc[0].to_dict()
        if not rep_rows.empty
        else {}
    )

    current_rep_actions = int(
        actions_per_rep.get(current_rep_id, 0)
    )

    # 6. Conversation
    conv_rows = conversation_df[
        conversation_df["opportunity_id"] == opportunity_id
    ]

    if conv_rows.empty:
        conversation_context = None
    else:
        row = conv_rows.iloc[0]
        conversation_context = {
            "email_text": row.get("email_text"),
            "call_notes": row.get("call_notes"),
        }

    # 7. Combined state
    result = {
        "opportunity_id": opp["opportunity_id"],
        "account_id": opp["account_id"],
        "industry": opp["industry"],
        "company_size": int(opp["company_size"]),
        "region": opp["region"],
        "stage": opp["stage"],
        "deal_value": float(opp["deal_value"]),
        "expected_close_days": int(
            opp["expected_close_days"]
        ),
        "current_products": list(
            opp["current_products"]
        ),
        "product_interest": list(
            opp["product_interest"]
        ),
        "engagement": engagement,
        "action_history": action_history,
        "rep_context": {
            "current_rep_id": current_rep_id,
            "experience_months": int(
                rep.get("experience_months", 0)
            ),
            "conversion_rate": float(
                rep.get("conversion_rate", 0.0)
            ),
            "active_opportunities": int(
                rep.get("active_opportunities", 0)
            ),
            "product_specialization": rep.get(
                "product_specialization"
            ),
            "num_reps_handled": num_reps_handled,
            "handoff_count": handoff_count,
            "current_rep_actions_on_opportunity":
                current_rep_actions,
            "actions_per_rep": {
                str(k): int(v)
                for k, v in actions_per_rep.items()
            },
        },
        "conversation_context": conversation_context,
    }

    # Validate before returning.
    return OpportunityState(
        **result
    ).model_dump()

In [ ]:
crm_data = get_opportunity_data(
    "OPP_1001"
)

print(
    "Opportunity:",
    crm_data["opportunity_id"],
)

print(
    "Last actions:",
    len(crm_data["action_history"]),
)

print(
    "Actions per rep:",
    crm_data["rep_context"]["actions_per_rep"],
)

print(
    "Handoffs:",
    crm_data["rep_context"]["handoff_count"],
)

crm_data

In [ ]:
@tool
def get_opportunity_tool(
    opportunity_id: str,
) -> dict:
    """
    Get latest structured CRM context for an opportunity.
    """
    return get_opportunity_data(
        opportunity_id
    )


@tool
def get_conversation_tool(
    opportunity_id: str,
) -> dict:
    """
    Get raw email/call history for an opportunity.
    """

    rows = conversation_df[
        conversation_df["opportunity_id"] == opportunity_id
    ]

    if rows.empty:
        return {
            "email_text": None,
            "call_notes": None,
        }

    row = rows.iloc[0]

    return {
        "email_text": row.get("email_text"),
        "call_notes": row.get("call_notes"),
    }

## Step 5.1 — Opportunity Risk Tool

In [ ]:
def predict_opportunity_risk(
    opportunity_data: dict,
) -> dict:
    """
    Demo risk model.
    Replace this function later with a trained model.
    """

    score = 0.15

    eng = opportunity_data["engagement"]
    rep = opportunity_data["rep_context"]

    if eng["website_visits_30d"] < 5:
        score += 0.25

    if eng["pricing_page_visits_30d"] == 0:
        score += 0.10

    if not eng["demo_requested"]:
        score += 0.05

    if opportunity_data["expected_close_days"] > 60:
        score += 0.20

    if rep["handoff_count"] >= 2:
        score += 0.10

    score = min(
        max(score, 0.0),
        1.0,
    )

    if score >= 0.70:
        level = "high"
    elif score >= 0.40:
        level = "medium"
    else:
        level = "low"

    return {
        "risk_score": round(score, 2),
        "risk_level": level,
    }


@tool
def opportunity_risk_tool(
    opportunity_id: str,
) -> dict:
    """
    Estimate risk that an opportunity will not progress/close.
    """

    data = get_opportunity_data(
        opportunity_id
    )

    if "error" in data:
        return data

    return predict_opportunity_risk(
        data
    )

## Step 5.2 — Retention / Uplift Tool

In [ ]:
def predict_retention_uplift(
    opportunity_data: dict,
) -> dict:
    """
    Demo uplift proxy.
    Replace later with a causal/uplift model.
    """

    score = 0.10

    eng = opportunity_data["engagement"]
    actions = opportunity_data["action_history"]
    rep = opportunity_data["rep_context"]

    if eng["pricing_page_visits_30d"] >= 2:
        score += 0.20

    if eng["website_visits_30d"] >= 8:
        score += 0.10

    if not eng["demo_requested"]:
        score += 0.10

    if rep["handoff_count"] >= 1:
        score += 0.10

    if actions:
        last_response = actions[-1].get(
            "response"
        )

        if last_response in {
            "opened",
            "clicked",
            "replied",
            "viewed",
        }:
            score += 0.15

    score = min(
        max(score, 0.0),
        1.0,
    )

    return {
        "uplift_score": round(score, 2),
        "intervention_needed": score >= 0.30,
    }


@tool
def retention_uplift_tool(
    opportunity_id: str,
) -> dict:
    """
    Estimate whether intervention is likely to help.
    """

    data = get_opportunity_data(
        opportunity_id
    )

    if "error" in data:
        return data

    return predict_retention_uplift(
        data
    )

## Step 5.3 — NBA Offline RL / IQL Tool

In [ ]:
ACTIONS = [
    "email",
    "linkedin_msg",
    "nurture",
    "call",
    "demo_schedule",
    "wait",
]


def predict_nba_iql(
    opportunity_data: dict,
) -> dict:
    """
    Demo replacement for the trained IQL policy.

    Production:
    lead/opportunity features
    + last 10 actions
    + gap embeddings
    + responses
    + rep context
        -> sequence encoder / GRU
        -> IQL
        -> action scores
    """

    scores = {
        "email": 0.40,
        "linkedin_msg": 0.30,
        "nurture": 0.35,
        "call": 0.45,
        "demo_schedule": 0.30,
        "wait": 0.20,
    }

    eng = opportunity_data["engagement"]
    actions = opportunity_data["action_history"]
    rep = opportunity_data["rep_context"]

    if eng["pricing_page_visits_30d"] >= 2:
        scores["call"] += 0.15
        scores["demo_schedule"] += 0.20

    if eng["website_visits_30d"] >= 10:
        scores["demo_schedule"] += 0.10

    if actions:
        last = actions[-1]

        if (
            last["action"] == "email"
            and last.get("response")
            in {"opened", "clicked", "replied"}
        ):
            scores["call"] += 0.10
            scores["demo_schedule"] += 0.10

        if last["action"] == "call":
            scores["call"] -= 0.20
            scores["wait"] += 0.15

        if (
            last["action"] == "linkedin_msg"
            and last.get("response") == "viewed"
        ):
            scores["email"] += 0.05
            scores["call"] += 0.05

    if rep["active_opportunities"] > 50:
        scores["call"] -= 0.10
        scores["demo_schedule"] -= 0.10
        scores["nurture"] += 0.10

    if rep["current_rep_actions_on_opportunity"] >= 5:
        scores["wait"] += 0.15

    scores = {
        action: round(
            min(max(value, 0.0), 1.0),
            2,
        )
        for action, value in scores.items()
    }

    recommended_action = max(
        scores,
        key=scores.get,
    )

    return {
        "recommended_action":
            recommended_action,
        "action_scores":
            scores,
    }


@tool
def nba_iql_tool(
    opportunity_id: str,
) -> dict:
    """
    Recommend next best action using the NBA/IQL interface.
    """

    data = get_opportunity_data(
        opportunity_id
    )

    if "error" in data:
        return data

    return predict_nba_iql(
        data
    )

## Step 5.4 — Rep Workload Tool

In [ ]:
def check_rep_workload(
    opportunity_data: dict,
) -> dict:
    """
    Simple current-rep capacity policy.
    """

    rep = opportunity_data[
        "rep_context"
    ]

    active = rep[
        "active_opportunities"
    ]

    if active >= 60:
        level = "high"
        available = False

    elif active >= 40:
        level = "medium"
        available = True

    else:
        level = "low"
        available = True

    return {
        "rep_id": rep["current_rep_id"],
        "active_opportunities": active,
        "workload_level": level,
        "available": available,
    }


@tool
def rep_workload_tool(
    opportunity_id: str,
) -> dict:
    """
    Check current assigned rep workload/availability.
    """

    data = get_opportunity_data(
        opportunity_id
    )

    if "error" in data:
        return data

    return check_rep_workload(
        data
    )

## Step 5.5 — Customer Adoption Trend Tool

This is a **deterministic analytics tool**, not an LLM agent.

### Why?

An LLM should not visually inspect 36 rows and guess whether usage is rising.

The tool:
1. reads the last 36 months of license + usage data
2. fits a simple linear trend
3. annualizes the slope
4. calculates recent year-over-year movement
5. returns a structured adoption signal

The Decision Agent consumes the result later.

### Production source

The notebook reads `license_usage_df`.

A deployed implementation would replace that read with a parameterized SQL query
against the governed licensing / telemetry source.

In [ ]:
# ============================================================
# CUSTOMER ADOPTION TREND
# ============================================================

TREND_THRESHOLD = 0.05
# ±5% annualized movement is treated as meaningful in this demo.
# Production thresholds should be calibrated from historical data.


def _annualized_linear_trend_pct(
    values: pd.Series,
) -> float:
    """
    Estimate annualized trend using a simple linear regression.

    Returns:
        Approximate annualized slope divided by the series mean.

    Example:
        +0.12 -> roughly +12% annualized direction
        -0.08 -> roughly -8% annualized direction
    """

    values = pd.to_numeric(
        values,
        errors="coerce",
    ).dropna()

    if len(values) < 2:
        return 0.0

    x = np.arange(
        len(values),
        dtype=float,
    )

    y = values.to_numpy(
        dtype=float
    )

    monthly_slope = np.polyfit(
        x,
        y,
        1,
    )[0]

    baseline = float(
        np.mean(
            np.abs(y)
        )
    )

    if baseline == 0:
        return 0.0

    return float(
        (monthly_slope * 12.0)
        / baseline
    )


def _classify_trend(
    annualized_pct: float,
) -> str:
    """
    Convert numeric slope into a stable business label.

    np.select keeps classification declarative rather than
    product/customer-specific nested branching.
    """

    label = np.select(
        [
            annualized_pct
            > TREND_THRESHOLD,

            annualized_pct
            < -TREND_THRESHOLD,
        ],
        [
            "increasing",
            "declining",
        ],
        default="stable",
    )

    return str(label)


def _recent_yoy_pct(
    values: pd.Series,
) -> Optional[float]:
    """
    Compare most recent 12 months against previous 12 months.

    Uses means so it works for both license count and usage tokens.
    """

    values = pd.to_numeric(
        values,
        errors="coerce",
    ).dropna()

    if len(values) < 24:
        return None

    previous_12m = float(
        values.iloc[-24:-12].mean()
    )

    latest_12m = float(
        values.iloc[-12:].mean()
    )

    if previous_12m == 0:
        return None

    return float(
        (latest_12m - previous_12m)
        / previous_12m
    )


def calculate_customer_adoption_trend(
    opportunity_id: str,
) -> CustomerAdoptionTrendOutput:
    """
    Production-shaped deterministic analytics function.

    The only notebook-specific part is reading from pandas.
    In production, replace that read with parameterized SQL.
    """

    opp_rows = opportunity_df[
        opportunity_df[
            "opportunity_id"
        ] == opportunity_id
    ]

    if opp_rows.empty:
        raise ValueError(
            f"Opportunity {opportunity_id} not found."
        )

    account_id = str(
        opp_rows.iloc[0][
            "account_id"
        ]
    )

    history = (
        license_usage_df[
            license_usage_df[
                "account_id"
            ] == account_id
        ]
        .sort_values(
            "month"
        )
        .tail(36)
        .copy()
    )

    if history.empty:
        raise ValueError(
            f"No adoption history for {account_id}."
        )

    license_slope = (
        _annualized_linear_trend_pct(
            history["license_count"]
        )
    )

    usage_slope = (
        _annualized_linear_trend_pct(
            history["usage_tokens"]
        )
    )

    license_trend = (
        _classify_trend(
            license_slope
        )
    )

    usage_trend = (
        _classify_trend(
            usage_slope
        )
    )

    # Overall business interpretation is a deterministic mapping.
    # This is not LLM reasoning.
    signal_map = {
        (
            "increasing",
            "increasing",
        ): "expanding",

        (
            "declining",
            "declining",
        ): "contracting",

        (
            "increasing",
            "declining",
        ): "underutilization_risk",

        (
            "declining",
            "increasing",
        ): "concentrated_usage",

        (
            "stable",
            "stable",
        ): "stable",
    }

    overall_signal = signal_map.get(
        (
            license_trend,
            usage_trend,
        ),
        "mixed",
    )

    latest = history.iloc[-1]

    latest_license_count = int(
        latest["license_count"]
    )

    latest_usage_tokens = float(
        latest["usage_tokens"]
    )

    latest_usage_per_license = (
        latest_usage_tokens
        / max(
            latest_license_count,
            1,
        )
    )

    return CustomerAdoptionTrendOutput(
        opportunity_id=opportunity_id,
        account_id=account_id,
        months_analyzed=len(
            history
        ),
        license_trend=license_trend,
        usage_trend=usage_trend,
        license_annualized_slope_pct=round(
            license_slope,
            4,
        ),
        usage_annualized_slope_pct=round(
            usage_slope,
            4,
        ),
        recent_license_yoy_pct=(
            None
            if (
                yoy := _recent_yoy_pct(
                    history[
                        "license_count"
                    ]
                )
            ) is None
            else round(
                yoy,
                4,
            )
        ),
        recent_usage_yoy_pct=(
            None
            if (
                yoy_usage := _recent_yoy_pct(
                    history[
                        "usage_tokens"
                    ]
                )
            ) is None
            else round(
                yoy_usage,
                4,
            )
        ),
        latest_license_count=(
            latest_license_count
        ),
        latest_usage_tokens=round(
            latest_usage_tokens,
            2,
        ),
        latest_usage_per_license=round(
            latest_usage_per_license,
            2,
        ),
        overall_adoption_signal=(
            overall_signal
        ),
    )


@tool
def customer_adoption_trend_tool(
    opportunity_id: str,
) -> dict:
    """
    Return deterministic 3-year customer license and usage trend signals.
    """

    return (
        calculate_customer_adoption_trend(
            opportunity_id
        )
        .model_dump()
    )


# Small standalone test.
customer_adoption_trend_tool.invoke({
    "opportunity_id": "OPP_1001"
})

## Step 5.5 — Autodesk Catalog RAG Tool — Local FAISS Version

We now use a **production-shaped RAG pipeline** instead of manual keyword scoring.

For this learning notebook, the catalog is still small sample data.
In production, the same pipeline would ingest approved Autodesk product
documentation from governed sources such as S3, a document repository,
or an internal knowledge platform.

### RAG flow

```text
Approved Autodesk catalog content
            ↓
Create document chunks + metadata
            ↓
OpenAI Embeddings
            ↓
Local FAISS vector index
            ↓
Customer requirement query
            ↓
Semantic similarity search
            ↓
Top relevant product evidence
            ↓
Product Fit Agent
```

### Why FAISS here?

FAISS gives us a real dense-vector similarity index locally without requiring
a managed vector database while we are learning and building the notebook.

The important design decision is that the agent calls the same tool:

```python
autodesk_catalog_rag_tool(query)
```

Later we can replace FAISS with OpenSearch Serverless without changing the
Product Fit Agent or LangGraph flow.

### Important production principle

The RAG tool should return **evidence + metadata**, not just a product name.

We therefore keep:

- product
- category
- content
- source
- section
- version
- approval status
- semantic similarity score

In [ ]:
# ============================================================
# AUTODESK PRODUCT CATALOG — SAMPLE TABULAR SOURCE
# ============================================================
#
# Think of this DataFrame as the output of:
#
#     SELECT ...
#     FROM approved_autodesk_product_catalog
#
# Each row becomes one LangChain Document.
# ============================================================

catalog_data = [
    {
        "chunk_id": "revit_001",
        "product": "Revit",
        "category": "AEC",
        "section": "BIM and multidisciplinary design",
        "version": "2026-demo",
        "source": "autodesk_catalog",
        "approved": True,
        "content": (
            "Revit supports Building Information Modeling workflows "
            "for architecture, engineering and construction. "
            "It supports coordinated multidisciplinary building design "
            "and documentation across architectural and structural disciplines."
        ),
    },
    {
        "chunk_id": "autocad_001",
        "product": "AutoCAD",
        "category": "CAD",
        "section": "General CAD workflows",
        "version": "2026-demo",
        "source": "autodesk_catalog",
        "approved": True,
        "content": (
            "AutoCAD supports general-purpose 2D drafting and 3D CAD "
            "workflows for technical drawings, documentation and design."
        ),
    },
    {
        "chunk_id": "fusion_001",
        "product": "Fusion",
        "category": "Manufacturing",
        "section": "Product design and manufacturing",
        "version": "2026-demo",
        "source": "autodesk_catalog",
        "approved": True,
        "content": (
            "Fusion supports product design, engineering and manufacturing "
            "workflows including 3D CAD and manufacturing activities."
        ),
    },
    {
        "chunk_id": "acc_001",
        "product": "Autodesk Construction Cloud",
        "category": "Construction",
        "section": "Project collaboration",
        "version": "2026-demo",
        "source": "autodesk_catalog",
        "approved": True,
        "content": (
            "Autodesk Construction Cloud supports project collaboration "
            "and information sharing across construction project teams."
        ),
    },
    {
        "chunk_id": "aec_collection_001",
        "product": "AEC Collection",
        "category": "AEC",
        "section": "AEC solution portfolio",
        "version": "2026-demo",
        "source": "autodesk_catalog",
        "approved": True,
        "content": (
            "The Architecture, Engineering and Construction Collection "
            "provides Autodesk tools for building, infrastructure and "
            "construction workflows requiring broader multidisciplinary capabilities."
        ),
    },
]


catalog_df = pd.DataFrame(
    catalog_data
)


# ============================================================
# TABULAR ROW -> LANGCHAIN DOCUMENT
# ============================================================

catalog_documents = []

for _, row in catalog_df.iterrows():

    if not bool(
        row["approved"]
    ):
        continue

    # Text embedded into vector space.
    page_content = f"""
Product: {row['product']}
Category: {row['category']}
Section: {row['section']}

Description:
{row['content']}
""".strip()

    # Metadata remains attached to the source document.
    metadata = {
        "chunk_id": row["chunk_id"],
        "product": row["product"],
        "category": row["category"],
        "section": row["section"],
        "version": row["version"],
        "source": row["source"],
    }

    catalog_documents.append(
        Document(
            page_content=page_content,
            metadata=metadata,
        )
    )


print(
    "Catalog rows:",
    len(catalog_df)
)

print(
    "Approved Documents:",
    len(catalog_documents)
)

catalog_documents[0]

In [ ]:
# ============================================================
# BUILD / LOAD LOCAL FAISS INDEX
# ============================================================

import hashlib
from pathlib import Path

import faiss
from langchain_openai import OpenAIEmbeddings


EMBEDDING_MODEL = "text-embedding-3-small"

FAISS_DIR = Path(
    "./local_faiss_autodesk_catalog"
)

FAISS_INDEX_FILE = (
    FAISS_DIR / "catalog.index"
)

FAISS_DOCUMENT_FILE = (
    FAISS_DIR / "documents.json"
)

FAISS_MANIFEST_FILE = (
    FAISS_DIR / "manifest.json"
)


def _serialize_documents(
    documents: List[Document],
) -> list[dict]:
    """
    Convert Documents into JSON-safe dictionaries.

    We intentionally persist text + metadata separately from FAISS.
    FAISS itself stores vectors, not our readable source documents.
    """

    return [
        {
            "page_content":
                doc.page_content,

            "metadata":
                doc.metadata,
        }
        for doc in documents
    ]


def _catalog_hash(
    documents: List[Document],
) -> str:
    """
    Fingerprint the approved catalog.

    If content or metadata changes, the hash changes.
    That causes a new index to be built.
    """

    payload = json.dumps(
        _serialize_documents(
            documents
        ),
        sort_keys=True,
    ).encode("utf-8")

    return hashlib.sha256(
        payload
    ).hexdigest()


CATALOG_HASH = _catalog_hash(
    catalog_documents
)


def _create_embedding_model():
    """
    OpenAI embedding client.

    Document embeddings are created only when building/rebuilding the index.
    Query embeddings are created at runtime.
    """

    if not os.getenv(
        "OPENAI_API_KEY"
    ):
        return None

    return OpenAIEmbeddings(
        model=EMBEDDING_MODEL
    )


def build_faiss_catalog_index(
    documents: List[Document],
    embedding_model,
):
    """
    Index approved catalog Documents using dense semantic embeddings.

    We use:
        IndexFlatIP + L2-normalized vectors

    This gives exact cosine-style similarity search for our small catalog.
    """

    texts = [
        doc.page_content
        for doc in documents
    ]

    vectors = (
        embedding_model
        .embed_documents(
            texts
        )
    )

    vectors = np.asarray(
        vectors,
        dtype="float32",
    )

    faiss.normalize_L2(
        vectors
    )

    dimension = int(
        vectors.shape[1]
    )

    index = faiss.IndexFlatIP(
        dimension
    )

    index.add(
        vectors
    )

    FAISS_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    faiss.write_index(
        index,
        str(
            FAISS_INDEX_FILE
        ),
    )

    FAISS_DOCUMENT_FILE.write_text(
        json.dumps(
            _serialize_documents(
                documents
            ),
            indent=2,
        ),
        encoding="utf-8",
    )

    FAISS_MANIFEST_FILE.write_text(
        json.dumps(
            {
                "catalog_hash":
                    CATALOG_HASH,

                "embedding_model":
                    EMBEDDING_MODEL,

                "document_count":
                    len(documents),
            },
            indent=2,
        ),
        encoding="utf-8",
    )

    return (
        index,
        documents,
    )


def load_or_build_faiss_catalog():
    """
    Reuse the local FAISS index if it matches the current catalog.

    Rebuild only when:
    - index files are missing
    - catalog content changed
    - embedding model changed
    """

    embedding_model = (
        _create_embedding_model()
    )

    if embedding_model is None:
        print(
            "OPENAI_API_KEY missing -> "
            "FAISS embeddings unavailable."
        )

        return (
            None,
            [],
            None,
        )

    files_exist = (
        FAISS_INDEX_FILE.exists()
        and FAISS_DOCUMENT_FILE.exists()
        and FAISS_MANIFEST_FILE.exists()
    )

    if files_exist:
        try:
            manifest = json.loads(
                FAISS_MANIFEST_FILE.read_text(
                    encoding="utf-8"
                )
            )

            is_current = (
                manifest.get(
                    "catalog_hash"
                )
                == CATALOG_HASH
                and manifest.get(
                    "embedding_model"
                )
                == EMBEDDING_MODEL
            )

            if is_current:
                index = faiss.read_index(
                    str(
                        FAISS_INDEX_FILE
                    )
                )

                raw_documents = json.loads(
                    FAISS_DOCUMENT_FILE.read_text(
                        encoding="utf-8"
                    )
                )

                documents = [
                    Document(
                        page_content=item[
                            "page_content"
                        ],
                        metadata=item[
                            "metadata"
                        ],
                    )
                    for item
                    in raw_documents
                ]

                print(
                    "Loaded existing FAISS catalog index."
                )

                return (
                    index,
                    documents,
                    embedding_model,
                )

        except Exception as exc:
            print(
                "Could not load existing FAISS index -> rebuilding."
            )
            print(
                "Reason:",
                type(exc).__name__,
            )

    print(
        "Building FAISS catalog index..."
    )

    index, documents = (
        build_faiss_catalog_index(
            catalog_documents,
            embedding_model,
        )
    )

    print(
        "FAISS catalog index created."
    )

    return (
        index,
        documents,
        embedding_model,
    )


(
    faiss_catalog_index,
    faiss_catalog_documents,
    catalog_embedding_model,
) = load_or_build_faiss_catalog()


RAG_MODE = (
    "faiss_semantic"
    if faiss_catalog_index
    is not None
    else "unavailable"
)

print(
    "RAG_MODE =",
    RAG_MODE,
)

### Step 5.5.2 — Semantic Retrieval from FAISS

At runtime we do **not rebuild the index**.

For each customer requirement:

```text
Customer requirement
        ↓
Create one query embedding
        ↓
FAISS similarity search
        ↓
Top-k catalog chunks
        ↓
Return content + source metadata
```

The result is cached using the catalog hash as part of the cache key.

That means:
- repeated identical searches can reuse retrieval results
- when catalog content changes, the catalog hash changes
- the old cached retrieval result is no longer used

This cache is appropriate for relatively stable product knowledge.
We do **not** apply the same pattern blindly to live CRM state.

In [ ]:
# ============================================================
# SEMANTIC SEARCH OVER FAISS
# ============================================================

@lru_cache(
    maxsize=256
)
def search_catalog_cached(
    query: str,
    k: int = 5,
    catalog_hash: str = CATALOG_HASH,
) -> tuple:
    """
    Retrieve semantically relevant approved Autodesk catalog Documents.

    Cache key contains:
        query + k + catalog hash

    If the catalog changes, CATALOG_HASH changes and stale retrieval
    results are automatically bypassed.
    """

    if (
        faiss_catalog_index is None
        or catalog_embedding_model is None
    ):
        return tuple()

    query_vector = (
        catalog_embedding_model
        .embed_query(
            query
        )
    )

    query_vector = np.asarray(
        [query_vector],
        dtype="float32",
    )

    faiss.normalize_L2(
        query_vector
    )

    safe_k = min(
        k,
        len(
            faiss_catalog_documents
        ),
    )

    if safe_k <= 0:
        return tuple()

    scores, positions = (
        faiss_catalog_index.search(
            query_vector,
            safe_k,
        )
    )

    results = []

    for score, position in zip(
        scores[0],
        positions[0],
    ):
        if position < 0:
            continue

        document = (
            faiss_catalog_documents[
                int(position)
            ]
        )

        results.append({
            "product":
                document.metadata[
                    "product"
                ],

            "category":
                document.metadata[
                    "category"
                ],

            "section":
                document.metadata[
                    "section"
                ],

            "version":
                document.metadata[
                    "version"
                ],

            "source":
                document.metadata[
                    "source"
                ],

            # The LLM receives readable evidence, not embedding vectors.
            "content":
                document.page_content,

            "semantic_score":
                round(
                    float(score),
                    4,
                ),
        })

    return tuple(
        json.dumps(
            item,
            sort_keys=True,
        )
        for item in results
    )


def search_catalog(
    query: str,
    k: int = 5,
) -> List[dict]:
    """
    Public local vector-search function.
    """

    return [
        json.loads(item)
        for item
        in search_catalog_cached(
            query,
            k,
            CATALOG_HASH,
        )
    ]


def retrieve_catalog(
    query: str,
    k: int = 5,
) -> List[dict]:
    """
    Stable retrieval interface.

    Today:
        local FAISS

    Later:
        OpenSearch / another managed vector backend

    The Product Fit Agent does not need to change.
    """

    return search_catalog(
        query=query,
        k=k,
    )


@tool
def autodesk_catalog_rag_tool(
    query: str,
) -> List[dict]:
    """
    Retrieve approved Autodesk catalog evidence relevant
    to the customer's actual workflow / business requirement.
    """

    return retrieve_catalog(
        query=query,
        k=5,
    )


# ------------------------------------------------------------
# Test RAG independently before involving the agent.
# ------------------------------------------------------------

faiss_test_results = retrieve_catalog(
    query=(
        "Customer needs BIM collaboration between "
        "architecture and structural teams."
    ),
    k=3,
)

faiss_test_results

### Step 5.5.3 — What Changes in Production?

The Product Fit Agent will still call:

```python
autodesk_catalog_rag_tool(query)
```

Only the implementation behind `retrieve_catalog()` changes.

```text
Notebook / local:
retrieve_catalog()
    ↓
FAISS

Production AWS:
retrieve_catalog()
    ↓
OpenSearch Serverless
```

This separation is intentional: **retrieval infrastructure is a tool/backend concern,
not an agent redesign**.

For a larger production catalog we would additionally add:

- real document loaders
- chunking strategy
- access-control metadata
- hybrid lexical + vector retrieval
- reranking
- document/version lifecycle
- retrieval evaluation
- monitoring

## Step 5.6 — Controlled Execution Tools

In [ ]:
# MOCK execution log.
# No real customer contact occurs.

EXECUTION_LOG: List[dict] = []


def _record_execution(
    item: dict,
) -> dict:
    EXECUTION_LOG.append(
        item
    )
    return item


@tool
def send_email_tool(
    opportunity_id: str,
    message: str,
) -> dict:
    """Mock email."""
    return _record_execution({
        "status": "success",
        "opportunity_id": opportunity_id,
        "action": "email",
        "detail": message,
    })


@tool
def send_linkedin_message_tool(
    opportunity_id: str,
    message: str,
) -> dict:
    """Mock LinkedIn message."""
    return _record_execution({
        "status": "success",
        "opportunity_id": opportunity_id,
        "action": "linkedin_msg",
        "detail": message,
    })


@tool
def create_call_task_tool(
    opportunity_id: str,
    rep_id: str,
    reason: str,
) -> dict:
    """Mock call task."""
    return _record_execution({
        "status": "success",
        "opportunity_id": opportunity_id,
        "action": "call",
        "rep_id": rep_id,
        "detail": reason,
    })


@tool
def schedule_demo_tool(
    opportunity_id: str,
    product: str,
) -> dict:
    """Mock demo scheduling."""
    return _record_execution({
        "status": "success",
        "opportunity_id": opportunity_id,
        "action": "demo_schedule",
        "product": product,
    })


@tool
def nurture_tool(
    opportunity_id: str,
    campaign: str,
) -> dict:
    """Mock nurture enrollment."""
    return _record_execution({
        "status": "success",
        "opportunity_id": opportunity_id,
        "action": "nurture",
        "campaign": campaign,
    })


@tool
def wait_tool(
    opportunity_id: str,
    reason: str,
) -> dict:
    """Record deliberate wait."""
    return _record_execution({
        "status": "success",
        "opportunity_id": opportunity_id,
        "action": "wait",
        "detail": reason,
    })


@tool
def update_crm_tool(
    opportunity_id: str,
    note: str,
) -> dict:
    """Mock CRM update."""
    return _record_execution({
        "status": "success",
        "opportunity_id": opportunity_id,
        "action": "crm_update",
        "detail": note,
    })

# Step 13 — Middleware Setup

We define middleware before creating the live agents so all agents can reuse it.

Middleware is instantiated only in real-LLM mode.

In [ ]:
AGENT_MIDDLEWARE = []

if USE_LLM_EFFECTIVE:
    try:
        from langchain.agents.middleware import (
            ModelRetryMiddleware,
            ToolRetryMiddleware,
            ModelCallLimitMiddleware,
            ToolCallLimitMiddleware,
            PIIMiddleware,
        )

        AGENT_MIDDLEWARE = [
            ModelRetryMiddleware(
                max_retries=2,
                initial_delay=0.5,
            ),
            ToolRetryMiddleware(
                max_retries=2,
                initial_delay=0.5,
            ),
            ModelCallLimitMiddleware(
                run_limit=8,
                exit_behavior="end",
            ),
            ToolCallLimitMiddleware(
                run_limit=12,
            ),
            PIIMiddleware(
                "email",
                strategy="redact",
                apply_to_input=True,
            ),
        ]

    except Exception as exc:
        print(
            "Live middleware unavailable -> continuing."
        )
        print(
            "Reason:",
            type(exc).__name__,
        )
        AGENT_MIDDLEWARE = []

print(
    "Middleware count:",
    len(AGENT_MIDDLEWARE),
)

# Step 6 — Opportunity Intelligence Agent

Strong LLM use case:

- Summarization
- Semantic intent extraction
- Pain-point extraction
- Objection extraction
- Requirement extraction
- Missing-information detection

In [ ]:
def demo_opportunity_intelligence(
    opportunity_id: str,
) -> OpportunityIntelligenceOutput:
    """
    Offline/debug fallback only.

    Production path:
        real LLM Opportunity Intelligence Agent.

    This fallback uses transparent keyword extraction only so that
    the notebook can still be debugged if the LLM is unavailable.
    """

    data = get_opportunity_data(
        opportunity_id
    )

    conversation = (
        data.get(
            "conversation_context"
        )
        or {}
    )

    raw_conversation = " ".join([
        str(
            conversation.get(
                "email_text"
            )
            or ""
        ),
        str(
            conversation.get(
                "call_notes"
            )
            or ""
        ),
    ])

    combined = (
        raw_conversation.lower()
    )

    pain_points = []
    objections = []
    requirements = []
    missing = []

    if (
        "coordination" in combined
        or "collaboration" in combined
    ):
        pain_points.append(
            "Cross-team coordination / collaboration difficulty"
        )

        requirements.append(
            "Improved multidisciplinary collaboration"
        )

    if "bim" in combined:
        requirements.append(
            "BIM workflow capability"
        )

    if (
        "pricing" in combined
        or "price" in combined
    ):
        objections.append(
            "Pricing concern"
        )

    timeline = (
        "next quarter"
        if "next quarter" in combined
        else None
    )

    if timeline is None:
        missing.append(
            "Purchase / decision timeline"
        )

    intent = (
        "high"
        if any(
            word in combined
            for word in [
                "evaluating",
                "decide",
                "decision",
                "pricing",
            ]
        )
        else (
            "medium"
            if combined.strip()
            else "unknown"
        )
    )

    if not combined.strip():
        missing.extend([
            "Customer pain points",
            "Customer objections",
            "Detailed workflow requirements",
        ])

    # Exact product-name detection only.
    # No product is inferred or hallucinated.
    known_products = {
        doc.metadata[
            "product"
        ]
        for doc in catalog_documents
    }

    mentioned_products = [
        product
        for product
        in sorted(
            known_products
        )
        if product.lower()
        in combined
    ]

    summary = (
        f"{data['industry']} opportunity in "
        f"{data['stage']} stage. "
        f"Current products: "
        f"{', '.join(data['current_products']) or 'unknown'}. "
        f"CRM product interest: "
        f"{', '.join(data['product_interest']) or 'unknown'}."
    )

    if raw_conversation.strip():
        summary += (
            " Conversation indicates a need around "
            + (
                "; ".join(
                    requirements
                )
                or "further discovery"
            )
            + "."
        )

    return OpportunityIntelligenceOutput(
        summary=summary,
        intent=intent,
        pain_points=pain_points,
        objections=objections,
        requirements=requirements,
        timeline=timeline,
        use_case=data.get(
            "industry"
        ),
        mentioned_products=(
            mentioned_products
        ),
        missing_information=missing,
    )

In [ ]:
OPPORTUNITY_INTELLIGENCE_PROMPT = """
You are an Autodesk Opportunity Intelligence Agent.

You must use the available tools to analyze:
1. structured CRM / opportunity information
2. raw email and call / CRM conversation text when available

Definitions:

- summary:
  Concise factual summary of the opportunity and customer conversation.
  Use only supplied information.

- intent:
  Strength of observable buying intent.
  high:
    clear evaluation, pricing, demo, timeline, stakeholder, or purchase signals.
  medium:
    customer shows interest but commitment/timeline is unclear.
  low:
    weak/exploratory interest.
  unknown:
    insufficient evidence.

- pain_points:
  Problems or difficulties the customer experiences today.
  Example:
  "Architecture and structural teams struggle to coordinate models."

- requirements:
  Desired capability, workflow, or business outcome.
  Example:
  "Needs BIM-based multidisciplinary coordination."

- objections:
  Barriers or concerns about purchase/adoption.
  Examples:
  pricing, migration, training, security, integration, budget.

- use_case:
  Main workflow/business scenario the customer wants to solve.

- timeline:
  Explicit decision, purchase, evaluation, or implementation timeframe.

- mentioned_products:
  Autodesk product names EXPLICITLY mentioned in the conversation.
  Do not infer a product name merely because a requirement sounds similar.
  If no product is explicitly mentioned, return [].

- missing_information:
  Important information required for a confident sales/product decision
  that is absent from the available data.

Rules:

1. Never invent facts.
2. Never invent a product name.
3. Keep pain points and requirements separate:
   pain point = current problem
   requirement = desired outcome/capability
4. Keep objections separate from pain points:
   objection = barrier to buying/adoption
5. Buying intent must be supported by observable evidence.
6. If conversation information is absent, say so through missing_information.
7. Do not recommend a product.
8. Do not choose the next sales action.
"""


opportunity_intelligence_agent = None

if USE_LLM_EFFECTIVE:
    try:
        opportunity_intelligence_agent = create_agent(
            model=f"openai:{OPENAI_MODEL}",
            tools=[
                get_opportunity_tool,
                get_conversation_tool,
            ],
            system_prompt=(
                OPPORTUNITY_INTELLIGENCE_PROMPT
            ),
            response_format=(
                OpportunityIntelligenceOutput
            ),
            middleware=AGENT_MIDDLEWARE,
        )

    except Exception as exc:
        print(
            "Could not create live Intelligence Agent."
        )
        print(
            "Reason:",
            type(exc).__name__,
        )


def run_opportunity_intelligence(
    opportunity_id: str,
) -> OpportunityIntelligenceOutput:
    """
    Safe wrapper:
    live agent -> fallback on failure.
    """

    if opportunity_intelligence_agent is not None:
        try:
            result = (
                opportunity_intelligence_agent
                .invoke({
                    "messages": [{
                        "role": "user",
                        "content": (
                            f"Analyze opportunity "
                            f"{opportunity_id}."
                        ),
                    }]
                })
            )

            return result[
                "structured_response"
            ]

        except Exception as exc:
            print(
                "Live intelligence failed -> fallback:",
                type(exc).__name__,
            )

    return demo_opportunity_intelligence(
        opportunity_id
    )


run_opportunity_intelligence(
    "OPP_1001"
)

# Step 7 — Product Fit & Solution Recommendation Agent

This agent answers:

> Does the customer's current / discussed Autodesk product fit the actual need?

## What goes into this agent?

```text
OpportunityIntelligenceOutput
    - summary
    - pain_points
    - requirements
    - objections
    - use_case
    - mentioned_products
    - missing_information

        +

CRM product context
    - current_products
    - product_interest
```

## Very important: embeddings are NOT sent to the LLM

There are two paths:

```text
OpportunityIntelligenceOutput
        │
        ├───────────────┐
        │               │
        ↓               │
Build retrieval query   │
        ↓               │
Embedding               │
        ↓               │
FAISS                   │
        ↓               │
Readable catalog docs   │
        │               │
        └──────┬────────┘
               ↓
        Product Fit LLM
               ↓
ProductRecommendationOutput
```

The numerical embedding vector is used only to search FAISS.

The LLM receives:
- structured customer intelligence
- CRM product context
- readable catalog evidence returned by FAISS

There are no hard-coded product mapping rules.

In [ ]:
def demo_product_recommendation(
    opportunity_id: str,
    intelligence: OpportunityIntelligenceOutput,
) -> ProductRecommendationOutput:
    """
    Offline/debug fallback only.

    Important:
    There are NO rules such as:
        if BIM -> Revit
        if manufacturing -> Fusion

    Candidate products come only from semantic FAISS retrieval.

    The real production path uses the Product Fit LLM to reason
    over retrieved catalog evidence.
    """

    opportunity = get_opportunity_data(
        opportunity_id
    )

    current_products = opportunity[
        "current_products"
    ]

    current_product_interest = opportunity[
        "product_interest"
    ]

    customer_need_parts = (
        intelligence.requirements
        + intelligence.pain_points
    )

    customer_need = (
        "; ".join(
            dict.fromkeys(
                customer_need_parts
            )
        )
        or "Customer need is not sufficiently defined."
    )

    # Build semantic retrieval text from Agent-1 output.
    retrieval_query = "\n".join([
        f"Summary: {intelligence.summary}",
        f"Use case: {intelligence.use_case or ''}",
        (
            "Requirements: "
            + "; ".join(
                intelligence.requirements
            )
        ),
        (
            "Pain points: "
            + "; ".join(
                intelligence.pain_points
            )
        ),
        (
            "Mentioned products: "
            + "; ".join(
                intelligence.mentioned_products
            )
        ),
    ])

    docs = retrieve_catalog(
        query=retrieval_query,
        k=5,
    )

    if not docs:
        return ProductRecommendationOutput(
            current_products=current_products,
            current_product_interest=current_product_interest,
            customer_need=customer_need,
            fit_status="NEEDS_DISCOVERY",
            recommended_product="UNKNOWN",
            reason=(
                "No grounded catalog evidence was retrieved."
            ),
            alternative_products=[],
            supporting_evidence=[],
            missing_information=list(
                intelligence.missing_information
            ),
            recommended_next_step=(
                "Collect additional workflow requirements."
            ),
        )

    best = docs[0]
    best_product = best[
        "product"
    ]

    discussed_products = {
        product.lower()
        for product in (
            current_products
            + current_product_interest
            + intelligence.mentioned_products
        )
    }

    # Generic fit check only.
    # Product candidate itself came from FAISS.
    fit_status = (
        "GOOD_FIT"
        if best_product.lower()
        in discussed_products
        else "POSSIBLE_MISMATCH"
    )

    return ProductRecommendationOutput(
        current_products=current_products,
        current_product_interest=current_product_interest,
        customer_need=customer_need,
        fit_status=fit_status,
        recommended_product=best_product,
        reason=(
            "Offline fallback selected the highest-ranked semantic "
            "catalog result. The live Product Fit Agent should make "
            "the final grounded fit judgment."
        ),
        alternative_products=[
            item["product"]
            for item in docs[1:3]
        ],
        supporting_evidence=[
            (
                f"{item['source']} | "
                f"{item['section']} | "
                f"{item['content']}"
            )
            for item in docs[:3]
        ],
        missing_information=list(
            intelligence.missing_information
        ),
        recommended_next_step=(
            "Validate the retrieved product fit with the customer."
        ),
    )

In [ ]:
PRODUCT_RECOMMENDATION_PROMPT = """
You are an Autodesk Product Fit & Solution Recommendation Agent.

Input context includes:
- Opportunity Intelligence summary
- pain points
- requirements
- objections
- use case
- products explicitly mentioned in conversation
- current products from CRM
- CRM product interest

Your job:

1. Understand the customer's actual workflow/business need.
2. Call autodesk_catalog_rag_tool using the customer's NEED,
   not merely a product name.
3. Read the returned catalog evidence.
4. Compare:
   - customer need
   - current product(s)
   - CRM product interest
   - products mentioned in the conversation
   - retrieved Autodesk product capabilities
5. Return:
   - GOOD_FIT
   - POSSIBLE_MISMATCH
   - NEEDS_DISCOVERY
6. Recommend a product only when supported by retrieved catalog evidence.
7. Explain why.
8. Return missing information and a recommended discovery next step.

Rules:

- MUST use autodesk_catalog_rag_tool.
- Never invent Autodesk product capabilities.
- Never invent a product name.
- Do not use a hard-coded product mapping.
- A POSSIBLE_MISMATCH is a discovery signal, not an automatic product switch.
- If requirements or evidence are insufficient, return NEEDS_DISCOVERY.
- Do not choose email/call/demo; that belongs to the Decision Agent.
"""


product_recommendation_agent = None

if USE_LLM_EFFECTIVE:
    try:
        product_recommendation_agent = create_agent(
            model=f"openai:{OPENAI_MODEL}",
            tools=[
                autodesk_catalog_rag_tool
            ],
            system_prompt=(
                PRODUCT_RECOMMENDATION_PROMPT
            ),
            response_format=(
                ProductRecommendationOutput
            ),
            middleware=AGENT_MIDDLEWARE,
        )

    except Exception as exc:
        print(
            "Could not create live Product Fit Agent."
        )
        print(
            "Reason:",
            type(exc).__name__,
        )


def run_product_recommendation(
    opportunity_id: str,
    intelligence: OpportunityIntelligenceOutput,
) -> ProductRecommendationOutput:
    """
    Safe live-agent/fallback wrapper.
    """

    opportunity = get_opportunity_data(
        opportunity_id
    )

    if "error" in opportunity:
        raise ValueError(
            opportunity["error"]
        )

    if product_recommendation_agent is not None:
        try:
            prompt = f"""
Analyze product fit for opportunity {opportunity_id}.

Current products:
{opportunity["current_products"]}

Current product interest:
{opportunity["product_interest"]}

Opportunity intelligence:
{intelligence.model_dump_json(indent=2)}

Use the Autodesk catalog tool.

Determine:
- GOOD_FIT
- POSSIBLE_MISMATCH
- NEEDS_DISCOVERY

If there is a mismatch, recommend a better-fit product only when
grounded in retrieved catalog evidence.
"""

            result = (
                product_recommendation_agent
                .invoke({
                    "messages": [{
                        "role": "user",
                        "content": prompt,
                    }]
                })
            )

            return result[
                "structured_response"
            ]

        except Exception as exc:
            print(
                "Live product fit agent failed -> fallback:",
                type(exc).__name__,
            )

    return demo_product_recommendation(
        opportunity_id,
        intelligence,
    )


_demo_intelligence = (
    run_opportunity_intelligence(
        "OPP_1001"
    )
)

run_product_recommendation(
    "OPP_1001",
    _demo_intelligence,
)

## Step 7.1 — Read the Product-Fit Result

The agent output now explicitly tells us:

```text
What product does the customer use?
What product are they asking about?
What are they actually trying to achieve?
Does the requested product fit?
What product is a better fit?
What should the rep do next?
```

The system does not automatically change the customer's product.
It gives the sales rep a grounded recommendation for the next discovery conversation.

In [ ]:
product_fit_demo = run_product_recommendation(
    "OPP_1001",
    run_opportunity_intelligence(
        "OPP_1001"
    ),
)

print(
    "Current products:",
    product_fit_demo.current_products,
)

print(
    "Current product interest:",
    product_fit_demo.current_product_interest,
)

print(
    "Customer need:",
    product_fit_demo.customer_need,
)

print(
    "Fit status:",
    product_fit_demo.fit_status,
)

print(
    "Recommended product:",
    product_fit_demo.recommended_product,
)

print(
    "Reason:",
    product_fit_demo.reason,
)

print(
    "Next step:",
    product_fit_demo.recommended_next_step,
)

# Step 8 — Decision / Sales Strategy Agent

This is the main reasoning agent.

It combines:

- Opportunity Intelligence
- Product Fit / Recommendation
- Opportunity Risk
- Retention / Uplift
- NBA-IQL
- Rep workload
- **3-year customer adoption trend**

### Why the adoption trend matters

Examples:

```text
Licenses ↑ + Usage ↑
→ healthy expansion signal

Licenses ↑ + Usage ↓
→ possible under-utilization / adoption issue

Licenses ↓ + Usage ↓
→ contraction / churn concern
```

The trend itself is calculated deterministically by
`customer_adoption_trend_tool`.

The LLM uses that structured signal while creating the final sales strategy.

NBA remains a signal, not an unconditional command.

In [ ]:
def demo_sales_strategy(
    opportunity_id: str,
    intelligence: OpportunityIntelligenceOutput,
    product: ProductRecommendationOutput,
) -> SalesStrategyOutput:
    """
    Deterministic fallback calling the same specialized tools.

    Product-fit result is one of the strategy inputs.
    """

    risk = opportunity_risk_tool.invoke({
        "opportunity_id": opportunity_id
    })

    uplift = retention_uplift_tool.invoke({
        "opportunity_id": opportunity_id
    })

    nba = nba_iql_tool.invoke({
        "opportunity_id": opportunity_id
    })

    workload = rep_workload_tool.invoke({
        "opportunity_id": opportunity_id
    })

    adoption = customer_adoption_trend_tool.invoke({
        "opportunity_id": opportunity_id
    })

    action = nba[
        "recommended_action"
    ]

    # If product fit is unclear, collect requirements
    # before doing a product-specific demo.
    if (
        product.fit_status == "NEEDS_DISCOVERY"
        and action == "demo_schedule"
    ):
        action = "call"

    # If the rep is unavailable, avoid a high-touch action.
    if (
        not workload["available"]
        and action in {
            "call",
            "demo_schedule",
        }
    ):
        action = "nurture"

    # If we found a likely product mismatch and customer intent is high,
    # a targeted demo/discovery session can be more useful than generic outreach.
    if (
        product.fit_status == "POSSIBLE_MISMATCH"
        and intelligence.intent == "high"
        and product.recommended_product != "UNKNOWN"
        and action in {
            "email",
            "linkedin_msg",
            "call",
        }
    ):
        action = "demo_schedule"

    approval = action in {
        "email",
        "linkedin_msg",
        "call",
        "demo_schedule",
    }

    reason = (
        f"NBA={nba['recommended_action']}; "
        f"risk={risk['risk_level']} ({risk['risk_score']}); "
        f"uplift={uplift['uplift_score']}; "
        f"rep_workload={workload['workload_level']}; "
        f"product_fit={product.fit_status}; "
        f"adoption={adoption['overall_adoption_signal']}; "
        f"license_trend={adoption['license_trend']}; "
        f"usage_trend={adoption['usage_trend']}."
    )

    talking_points = list(
        dict.fromkeys(
            intelligence.pain_points
            + intelligence.requirements
            + [
                f"Product fit status: {product.fit_status}",
                f"Recommended solution: {product.recommended_product}",
                product.reason,
            ]
        )
    )

    questions = list(
        intelligence.missing_information
    )

    questions.extend(
        product.missing_information
    )

    questions = list(
        dict.fromkeys(
            questions
        )
    )

    if not questions:
        questions = [
            "Who are the technical and commercial stakeholders?",
            "What is the customer's exact decision process?",
        ]

    return SalesStrategyOutput(
        recommended_action=action,
        recommended_product=(
            product.recommended_product
        ),
        strategy=(
            f"Use a {action} motion. "
            f"Product-fit status is {product.fit_status}. "
            f"Position {product.recommended_product} only when supported "
            f"by the customer's stated workflow and catalog evidence."
        ),
        reason=reason,
        talking_points=talking_points,
        discovery_questions=questions,
        requires_human_approval=approval,
    )

In [ ]:
DECISION_AGENT_PROMPT = """
You are an Autodesk Sales Strategy Agent.

Inputs:
- Opportunity intelligence
- Grounded product recommendation

Tools:
- CRM
- Opportunity risk
- Retention/uplift
- NBA-IQL
- Rep workload
- Customer 3-year license / usage adoption trend

Tasks:
1. Check tool signals.
2. Treat NBA as one signal.
3. Select the final sales action.
4. Produce strategy, reason, talking points and discovery questions.

Rules:
- Never invent facts/products.
- Do not execute the action.
"""


decision_agent = None

if USE_LLM_EFFECTIVE:
    try:
        decision_agent = create_agent(
            model=f"openai:{OPENAI_MODEL}",
            tools=[
                get_opportunity_tool,
                opportunity_risk_tool,
                retention_uplift_tool,
                nba_iql_tool,
                rep_workload_tool,
                customer_adoption_trend_tool,
            ],
            system_prompt=(
                DECISION_AGENT_PROMPT
            ),
            response_format=(
                SalesStrategyOutput
            ),
            middleware=AGENT_MIDDLEWARE,
        )

    except Exception as exc:
        print(
            "Could not create live Decision Agent."
        )
        print(
            "Reason:",
            type(exc).__name__,
        )


def run_sales_strategy(
    opportunity_id: str,
    intelligence: OpportunityIntelligenceOutput,
    product: ProductRecommendationOutput,
) -> SalesStrategyOutput:
    """
    Safe live-agent/fallback wrapper.
    """

    if decision_agent is not None:
        try:
            prompt = (
                f"Create the best strategy for "
                f"{opportunity_id}.\n\n"
                f"INTELLIGENCE:\n"
                f"{intelligence.model_dump_json(indent=2)}\n\n"
                f"PRODUCT:\n"
                f"{product.model_dump_json(indent=2)}"
            )

            result = decision_agent.invoke({
                "messages": [{
                    "role": "user",
                    "content": prompt,
                }]
            })

            return result[
                "structured_response"
            ]

        except Exception as exc:
            print(
                "Live decision failed -> fallback:",
                type(exc).__name__,
            )

    return demo_sales_strategy(
        opportunity_id,
        intelligence,
        product,
    )


_demo_product = run_product_recommendation(
    "OPP_1001",
    _demo_intelligence
)

run_sales_strategy(
    "OPP_1001",
    _demo_intelligence,
    _demo_product,
)

# Step 9 — Action Agent

The Action Agent receives an already-approved strategy.

It cannot re-run NBA or change the recommended product/action.

In [ ]:
def demo_execute_action(
    opportunity_id: str,
    strategy: SalesStrategyOutput,
) -> ActionAgentOutput:
    """
    Safe deterministic dispatcher.
    """

    data = get_opportunity_data(
        opportunity_id
    )

    if "error" in data:
        return ActionAgentOutput(
            action=strategy.recommended_action,
            status="failed",
            message=data["error"],
        )

    rep_id = data[
        "rep_context"
    ][
        "current_rep_id"
    ]

    action = strategy[
        "recommended_action"
    ] if isinstance(
        strategy,
        dict,
    ) else strategy.recommended_action

    if action == "email":
        result = send_email_tool.invoke({
            "opportunity_id": opportunity_id,
            "message": strategy.strategy,
        })

    elif action == "linkedin_msg":
        result = (
            send_linkedin_message_tool
            .invoke({
                "opportunity_id":
                    opportunity_id,
                "message":
                    strategy.strategy,
            })
        )

    elif action == "call":
        result = create_call_task_tool.invoke({
            "opportunity_id": opportunity_id,
            "rep_id": rep_id,
            "reason": strategy.reason,
        })

    elif action == "demo_schedule":
        result = schedule_demo_tool.invoke({
            "opportunity_id": opportunity_id,
            "product":
                strategy.recommended_product,
        })

    elif action == "nurture":
        result = nurture_tool.invoke({
            "opportunity_id": opportunity_id,
            "campaign": (
                f"{strategy.recommended_product}"
                f"_nurture"
            ),
        })

    elif action == "wait":
        result = wait_tool.invoke({
            "opportunity_id": opportunity_id,
            "reason": strategy.reason,
        })

    else:
        return ActionAgentOutput(
            action=action,
            status="failed",
            message=(
                f"Unsupported action: {action}"
            ),
        )

    update_crm_tool.invoke({
        "opportunity_id": opportunity_id,
        "note": (
            f"Executed {action}. "
            f"Product="
            f"{strategy.recommended_product}. "
            f"Reason={strategy.reason}"
        ),
    })

    return ActionAgentOutput(
        action=action,
        status=(
            "success"
            if result.get("status") == "success"
            else "failed"
        ),
        message=(
            f"Executed {action} and "
            f"recorded the result in mock CRM."
        ),
    )

In [ ]:
ACTION_AGENT_PROMPT = """
You are an Autodesk Sales Action Agent.

You receive an already-approved strategy.

Rules:
1. Execute ONLY the recommended action.
2. Do not change action/product/strategy.
3. Do not call NBA/Risk/RAG.
4. Use the matching execution tool.
5. Update CRM after successful execution.
"""


action_agent = None

if USE_LLM_EFFECTIVE:
    try:
        action_agent = create_agent(
            model=f"openai:{OPENAI_MODEL}",
            tools=[
                send_email_tool,
                send_linkedin_message_tool,
                create_call_task_tool,
                schedule_demo_tool,
                nurture_tool,
                wait_tool,
                update_crm_tool,
            ],
            system_prompt=ACTION_AGENT_PROMPT,
            response_format=ActionAgentOutput,
            middleware=AGENT_MIDDLEWARE,
        )

    except Exception as exc:
        print(
            "Could not create live Action Agent."
        )
        print(
            "Reason:",
            type(exc).__name__,
        )


def run_action_agent(
    opportunity_id: str,
    strategy: SalesStrategyOutput,
) -> ActionAgentOutput:
    """
    Safe live-agent/fallback wrapper.
    """

    if action_agent is not None:
        try:
            data = get_opportunity_data(
                opportunity_id
            )

            current_rep = data[
                "rep_context"
            ][
                "current_rep_id"
            ]

            prompt = (
                f"Execute the approved strategy.\n"
                f"Opportunity={opportunity_id}\n"
                f"Current rep={current_rep}\n\n"
                f"{strategy.model_dump_json(indent=2)}"
            )

            result = action_agent.invoke({
                "messages": [{
                    "role": "user",
                    "content": prompt,
                }]
            })

            return result[
                "structured_response"
            ]

        except Exception as exc:
            print(
                "Live action failed -> fallback:",
                type(exc).__name__,
            )

    return demo_execute_action(
        opportunity_id,
        strategy,
    )

# Step 10 — LangGraph State and Nodes

We store plain dictionaries in graph state for easy checkpoint serialization.

In [ ]:
class SalesGraphState(TypedDict):
    opportunity_id: str
    intelligence: Optional[dict]
    product_recommendation: Optional[dict]
    sales_strategy: Optional[dict]
    guardrail_result: Optional[dict]
    approval_status: Optional[str]
    execution_result: Optional[dict]


def opportunity_intelligence_node(
    state: SalesGraphState,
) -> dict:
    output = run_opportunity_intelligence(
        state["opportunity_id"]
    )

    return {
        "intelligence":
            output.model_dump()
    }


def product_recommendation_node(
    state: SalesGraphState,
) -> dict:
    intelligence = OpportunityIntelligenceOutput(
        **state["intelligence"]
    )

    output = run_product_recommendation(
        state["opportunity_id"],
        intelligence
    )

    return {
        "product_recommendation":
            output.model_dump()
    }


def decision_node(
    state: SalesGraphState,
) -> dict:
    intelligence = OpportunityIntelligenceOutput(
        **state["intelligence"]
    )

    product = ProductRecommendationOutput(
        **state["product_recommendation"]
    )

    output = run_sales_strategy(
        state["opportunity_id"],
        intelligence,
        product,
    )

    return {
        "sales_strategy":
            output.model_dump()
    }

# Step 15 — Guardrails

We define deterministic guardrails before graph routing.

These are not LLM decisions.

In [ ]:
ALLOWED_PRODUCTS = {
    doc.metadata["product"]
    for doc in catalog_documents
}

ALLOWED_ACTIONS = set(
    ACTIONS
)


def run_guardrails(
    opportunity_id: str,
    strategy: SalesStrategyOutput,
) -> GuardrailResult:
    """
    Deterministic business/safety checks.
    """

    reasons = []
    passed = True
    force_approval = False

    if (
        strategy.recommended_product
        not in ALLOWED_PRODUCTS
    ):
        passed = False
        reasons.append(
            "Product is not grounded in approved catalog."
        )

    if (
        strategy.recommended_action
        not in ALLOWED_ACTIONS
    ):
        passed = False
        reasons.append(
            "Action is not in allowed action set."
        )

    workload = rep_workload_tool.invoke({
        "opportunity_id":
            opportunity_id
    })

    if (
        not workload.get(
            "available",
            True,
        )
        and strategy.recommended_action
        in {"call", "demo_schedule"}
    ):
        force_approval = True
        reasons.append(
            "Rep unavailable for high-touch action."
        )

    # Prototype policy:
    # external/high-touch actions require approval.
    if strategy.recommended_action in {
        "email",
        "linkedin_msg",
        "call",
        "demo_schedule",
    }:
        force_approval = True

    if not reasons:
        reasons.append(
            "All deterministic guardrails passed."
        )

    return GuardrailResult(
        passed=passed,
        reasons=reasons,
        force_human_approval=(
            force_approval
        ),
    )


def guardrail_node(
    state: SalesGraphState,
) -> dict:
    strategy = SalesStrategyOutput(
        **state["sales_strategy"]
    )

    result = run_guardrails(
        state["opportunity_id"],
        strategy,
    )

    return {
        "guardrail_result":
            result.model_dump()
    }

# Step 11 — Conditional Routing + Human Approval

- Failed guardrail → END
- Approval needed → Human Approval
- No approval needed → Action

In [ ]:
def route_after_guardrails(
    state: SalesGraphState,
):
    guard = GuardrailResult(
        **state["guardrail_result"]
    )

    strategy = SalesStrategyOutput(
        **state["sales_strategy"]
    )

    if not guard.passed:
        return END

    if (
        guard.force_human_approval
        or strategy.requires_human_approval
    ):
        return "human_approval"

    return "action"


def human_approval_node(
    state: SalesGraphState,
) -> dict:
    """
    Demo mode auto-approves.
    HITL mode pauses using interrupt().
    """

    strategy = SalesStrategyOutput(
        **state["sales_strategy"]
    )

    if not ENABLE_HITL:
        return {
            "approval_status":
                "approved"
        }

    approved = interrupt({
        "message":
            "Human approval required",
        "opportunity_id":
            state["opportunity_id"],
        "recommended_action":
            strategy.recommended_action,
        "recommended_product":
            strategy.recommended_product,
        "strategy":
            strategy.strategy,
        "reason":
            strategy.reason,
        "question":
            "Approve this action?",
    })

    return {
        "approval_status": (
            "approved"
            if bool(approved)
            else "rejected"
        )
    }


def route_after_approval(
    state: SalesGraphState,
):
    if (
        state.get("approval_status")
        == "approved"
    ):
        return "action"

    return END


def action_node(
    state: SalesGraphState,
) -> dict:
    strategy = SalesStrategyOutput(
        **state["sales_strategy"]
    )

    result = run_action_agent(
        state["opportunity_id"],
        strategy,
    )

    return {
        "execution_result":
            result.model_dump()
    }

# Step 12 — Memory / Checkpointing

LangGraph checkpointer stores the workflow state.

CRM/SQL remains the source of truth for business facts.

In [ ]:
checkpointer = InMemorySaver()

print(
    "Checkpointer:",
    type(checkpointer).__name__,
)

# Step 13 — Middleware

Already configured before the agents.

In real-LLM mode we use:

- Model retry
- Tool retry
- Model-call limit
- Tool-call limit
- PII email redaction

This prevents each individual agent from implementing its own retry/safety boilerplate.

In [ ]:
if AGENT_MIDDLEWARE:
    for item in AGENT_MIDDLEWARE:
        print(
            type(item).__name__
        )
else:
    print(
        "Demo mode: live middleware not instantiated."
    )

logging.basicConfig(
    level=logging.INFO,
    format=(
        "%(asctime)s | "
        "%(levelname)s | "
        "%(message)s"
    ),
)

logger = logging.getLogger(
    "autodesk_sales_graph"
)

# Step 14 — Caching

We cache only where the data is sufficiently stable.

## 1. FAISS index persistence

The product catalog is embedded once and saved locally:

```text
catalog Documents
    ↓
document embeddings
    ↓
FAISS index
    ↓
catalog.index
```

We do **not** rebuild document embeddings for every opportunity.

A catalog hash detects content changes and triggers index rebuilding.

---

## 2. Retrieval query cache

`search_catalog_cached()` caches:

```text
query + k + CATALOG_HASH
```

So repeated identical semantic searches can avoid another query-embedding/search call.

When the catalog changes:
- `CATALOG_HASH` changes
- previous retrieval-cache keys no longer match

---

## 3. What we do NOT long-term cache

Do not blindly cache:
- current opportunity stage
- latest customer conversation
- rep workload
- current engagement
- recent actions
- latest license / usage telemetry

The 3-year adoption tool is therefore read fresh in this notebook.

In a deployed service, a short TTL cache may be reasonable for expensive
warehouse queries, but the cache key should include an `as_of` / source-version
signal so stale telemetry is not silently reused.

In [ ]:
print(
    "Before:",
    search_catalog_cached.cache_info(),
)

_ = search_catalog(
    "BIM multidisciplinary architecture and structural coordination",
    k=3,
)

_ = search_catalog(
    "BIM multidisciplinary architecture and structural coordination",
    k=3,
)

print(
    "After:",
    search_catalog_cached.cache_info(),
)

# Step 10 continued — Compile the Final LangGraph

In [ ]:
builder = StateGraph(
    SalesGraphState
)

builder.add_node(
    "opportunity_intelligence",
    opportunity_intelligence_node,
)

builder.add_node(
    "product_recommendation",
    product_recommendation_node,
)

builder.add_node(
    "decision",
    decision_node,
)

builder.add_node(
    "guardrails",
    guardrail_node,
)

builder.add_node(
    "human_approval",
    human_approval_node,
)

builder.add_node(
    "action",
    action_node,
)

builder.add_edge(
    START,
    "opportunity_intelligence",
)

builder.add_edge(
    "opportunity_intelligence",
    "product_recommendation",
)

builder.add_edge(
    "product_recommendation",
    "decision",
)

builder.add_edge(
    "decision",
    "guardrails",
)

builder.add_conditional_edges(
    "guardrails",
    route_after_guardrails,
)

builder.add_conditional_edges(
    "human_approval",
    route_after_approval,
)

builder.add_edge(
    "action",
    END,
)

sales_graph = builder.compile(
    checkpointer=checkpointer
)

print(
    "LangGraph compiled successfully."
)

## Optional graph image

If Mermaid rendering is unavailable, the cell prints the flow instead of failing.

In [ ]:
try:
    from IPython.display import (
        Image,
        display,
    )

    display(
        Image(
            sales_graph
            .get_graph()
            .draw_mermaid_png()
        )
    )

except Exception:
    print(
        "START -> Intelligence -> Product "
        "-> Decision -> Guardrails "
        "-> Approval/Action -> END"
    )

# Step 16 — End-to-End Run

Only business input:

```text
opportunity_id
```

The graph retrieves and generates the rest.

In [ ]:
def initial_graph_state(
    opportunity_id: str,
) -> SalesGraphState:
    """
    Clean initial graph state.
    """

    return {
        "opportunity_id":
            opportunity_id,
        "intelligence":
            None,
        "product_recommendation":
            None,
        "sales_strategy":
            None,
        "guardrail_result":
            None,
        "approval_status":
            None,
        "execution_result":
            None,
    }


config = {
    "configurable": {
        "thread_id":
            "OPP_1001_RUN_1"
    }
}

final_state = sales_graph.invoke(
    initial_graph_state(
        "OPP_1001"
    ),
    config=config,
)

print(
    "Workflow finished."
)

In [ ]:
print(
    "\n=== INTELLIGENCE ==="
)
print(
    json.dumps(
        final_state.get(
            "intelligence"
        ),
        indent=2,
    )
)

print(
    "\n=== PRODUCT ==="
)
print(
    json.dumps(
        final_state.get(
            "product_recommendation"
        ),
        indent=2,
    )
)

print(
    "\n=== STRATEGY ==="
)
print(
    json.dumps(
        final_state.get(
            "sales_strategy"
        ),
        indent=2,
    )
)

print(
    "\n=== GUARDRAILS ==="
)
print(
    json.dumps(
        final_state.get(
            "guardrail_result"
        ),
        indent=2,
    )
)

print(
    "\n=== APPROVAL ==="
)
print(
    final_state.get(
        "approval_status"
    )
)

print(
    "\n=== EXECUTION ==="
)
print(
    json.dumps(
        final_state.get(
            "execution_result"
        ),
        indent=2,
    )
)

## Inspect checkpoint memory

In [ ]:
snapshot = sales_graph.get_state(
    config
)

print(
    "Opportunity:",
    snapshot.values.get(
        "opportunity_id"
    )
)

print(
    "Next nodes:",
    snapshot.next,
)

history = list(
    sales_graph.get_state_history(
        config
    )
)

print(
    "Checkpoint count:",
    len(history),
)

## Optional real human-in-the-loop test

Set:

```python
ENABLE_HITL = True
```

Then use a new thread ID.

```python
hitl_config = {
    "configurable": {
        "thread_id": "OPP_1001_HITL_1"
    }
}

paused = sales_graph.invoke(
    initial_graph_state("OPP_1001"),
    config=hitl_config,
)

# Approve
resumed = sales_graph.invoke(
    Command(resume=True),
    config=hitl_config,
)

# Or reject
# resumed = sales_graph.invoke(
#     Command(resume=False),
#     config=hitl_config,
# )
```

# Step 17 — Evaluation / Smoke Tests

We test the data, tools, agent contracts, grounding and cold/no-conversation path.

In [ ]:
def run_smoke_tests() -> pd.DataFrame:
    """
    Lightweight notebook tests.
    """

    tests = []

    def check(
        name: str,
        condition: bool,
        detail: str = "",
    ):
        tests.append({
            "test": name,
            "passed": bool(condition),
            "detail": detail,
        })

    # CRM
    data = get_opportunity_data(
        "OPP_1001"
    )

    check(
        "CRM opportunity found",
        data.get("opportunity_id")
        == "OPP_1001",
    )

    check(
        "Action history <= 10",
        len(
            data.get(
                "action_history",
                [],
            )
        ) <= 10,
    )

    check(
        "Actions-per-rep exists",
        bool(
            data["rep_context"][
                "actions_per_rep"
            ]
        ),
    )

    validated = OpportunityState(
        **data
    )

    check(
        "OpportunityState validates",
        validated.opportunity_id
        == "OPP_1001",
    )

    # NBA
    nba = nba_iql_tool.invoke({
        "opportunity_id":
            "OPP_1001"
    })

    check(
        "NBA action allowed",
        nba["recommended_action"]
        in ALLOWED_ACTIONS,
        nba["recommended_action"],
    )


    # 3-year customer adoption trend
    adoption = customer_adoption_trend_tool.invoke({
        "opportunity_id":
            "OPP_1001"
    })

    check(
        "Adoption history is 36 months",
        adoption["months_analyzed"]
        == 36,
        str(
            adoption["months_analyzed"]
        ),
    )

    check(
        "Adoption signal valid",
        adoption[
            "overall_adoption_signal"
        ]
        in {
            "expanding",
            "stable",
            "contracting",
            "underutilization_risk",
            "concentrated_usage",
            "mixed",
        },
        adoption[
            "overall_adoption_signal"
        ],
    )

    # RAG
    docs = retrieve_catalog(
        "architecture BIM collaboration",
        k=3,
    )

    check(
        "Catalog returns results",
        len(docs) > 0,
    )

    check(
        "Catalog results grounded",
        all(
            item["product"]
            in ALLOWED_PRODUCTS
            for item in docs
        ),
    )

    # Agents
    intel = run_opportunity_intelligence(
        "OPP_1001"
    )

    product = run_product_recommendation(
        "OPP_1001",
        intel
    )

    strategy = run_sales_strategy(
        "OPP_1001",
        intel,
        product,
    )

    check(
        "Intelligence schema valid",
        isinstance(
            intel,
            OpportunityIntelligenceOutput,
        ),
    )

    check(
        "Product grounded",
        product.recommended_product
        in ALLOWED_PRODUCTS,
        product.recommended_product,
    )

    check(
        "Product fit status valid",
        product.fit_status
        in {
            "GOOD_FIT",
            "POSSIBLE_MISMATCH",
            "NEEDS_DISCOVERY",
        },
        product.fit_status,
    )

    check(
        "Strategy action allowed",
        strategy.recommended_action
        in ALLOWED_ACTIONS,
        strategy.recommended_action,
    )

    guard = run_guardrails(
        "OPP_1001",
        strategy,
    )

    check(
        "Guardrail schema valid",
        isinstance(
            guard,
            GuardrailResult,
        ),
    )

    # Cold / no-conversation path
    cold = run_opportunity_intelligence(
        "OPP_1002"
    )

    check(
        "No-conversation path works",
        isinstance(
            cold,
            OpportunityIntelligenceOutput,
        ),
        cold.intent,
    )

    return pd.DataFrame(
        tests
    )


test_results = run_smoke_tests()

test_results

In [ ]:
failed = test_results[
    ~test_results["passed"]
]

if failed.empty:
    print(
        "✅ All smoke tests passed."
    )
else:
    print(
        "❌ Some smoke tests failed:"
    )
    display(
        failed
    )

# Final Architecture Summary

```text
                         opportunity_id
                              │
                              ▼
                  Opportunity Intelligence Agent
                    │                    │
                 CRM Tool        Conversation Tool
                    └─────────┬──────────┘
                              ▼
                     Structured Intelligence
                              │
                              ▼
                    Product Fit & Solution Recommendation Agent
                              │
                    Autodesk Catalog RAG
                              │
                              ▼
                      Grounded Product
                              │
                              ▼
                      Decision / Strategy Agent
             ┌────────────────┼─────────────────┐
             ▼                ▼                 ▼
         Risk Tool       Uplift Tool        NBA IQL Tool
                                                │
                                        Rep Workload Tool
                              │
                              ▼
                         Sales Strategy
                              │
                              ▼
                    Deterministic Guardrails
                              │
                     Human approval if needed
                              │
                              ▼
                         Action Agent
                              │
         Email / LinkedIn / Call / Demo / Nurture / Wait
                              │
                              ▼
                             CRM
```

## What stays non-LLM?

- Risk model
- Uplift model
- NBA/IQL policy
- SQL/CRM
- Rep workload
- Deterministic guardrails
- External API execution

That separation is deliberate.

# Optional Step 18 — Production Extensions

After the notebook is understood and stable:

1. Connect real Salesforce/SQL data.
2. Replace mock risk with trained model service.
3. Replace mock uplift with causal/uplift model.
4. Load real IQL checkpoint.
5. Ingest governed Autodesk catalog content.
6. Use a production vector database.
7. Use durable LangGraph checkpoint storage.
8. Add tracing/observability.
9. Add stronger RBAC and tool authorization.
10. Add FastAPI.
11. Containerize.
12. Deploy to AWS.
13. Add monitoring + controlled A/B rollout.